# 03a — Video Mode 1: First/Last(/Middle)-Frame → Video (LTX-Video 0.9.8-13B-distilled)

Replaces the ComfyUI LTX-Video FLF2V workflow (`03_video_mode1_ltx_flf2v.json`). Pure diffusers:
`LTXConditionPipeline` + `LTXVideoCondition` — no custom nodes, no server.

**What it does:** takes a start still (required), an end still (optional → plain I2V), and optional
middle keyframes, then generates the video between them. This is the workhorse for Mode 3 (agentic)
scene interpolation.

**Why this model (spec 03 default):** Apache-2.0, guidance/timestep-distilled (8 steps + 5-step
upscale refine pass ≈ a 13B-quality clip in a couple of minutes on A100), supports *multiple*
keyframes in one pass (ComfyUI's `LTXVAddGuide` did the same thing).

**Inputs** (paths in Drive, from 02a stills or uploaded here):
- `start frame` — required still of the character
- `end frame` — optional target still
- `middle frames` — optional `{path, fraction}` list, fraction ∈ (0,1) of clip length

**VRAM:** 13B distilled ≈ 26 GB bf16 + VAE decode. Strategy auto-selected; 40 GB A100 uses group
offloading, 80 GB runs resident. T4/free tier → see the commented 2B / GGUF variants in §8.

## 1. Config + mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'Yuna'
TRIGGER_TOKEN  = 'sks_vyuna'
NUM_FRAMES     = 121     # 8k+1 frames; 121 ≈ 5 s @ 24 fps
FPS            = 24
WIDTH, HEIGHT  = 832, 480   # landscape 16:9 LTX default; 480x832 portrait also fine
UPSCALE_2X     = True     # latent 2× upscale + 5-step refine pass (docs recipe)
# ─────────────────────────────────────────────────────────────────────────

import os
DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
VID_OUT    = f'{DRIVE_BASE}/outputs/videos/{CHARACTER_NAME}/mode1_ltx'
os.makedirs(VID_OUT, exist_ok=True)
os.environ['HF_HOME'] = f'{DRIVE_BASE}/models/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

# Where your keyframes live (02a output dirs) — or set absolute paths below in §6
STILLS_ROOT = f'{DRIVE_BASE}/outputs/images/{CHARACTER_NAME}'
print(f'Video out : {VID_OUT}')
print(f'Stills root: {STILLS_ROOT}')

## 2. HuggingFace login + install (uv)
LTX-Video is Apache-2.0 — no gated license, token is only needed if you use a private cache. We still
pass the Colab Secrets token for consistency.

In [ ]:
import os
hf_token = ''
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
os.environ.setdefault('HF_TOKEN', hf_token)

!pip install -q uv
# diffusers must be recent enough for LTX 0.9.8 + latent upsample pipeline
!uv pip install --system -q --reinstall-package diffusers \
    "diffusers>=0.35.0" transformers accelerate safetensors huggingface_hub

import torch, diffusers
print('torch', torch.__version__, '| diffusers', diffusers.__version__)

## 3. Load LTX-Video 0.9.8-13B-distilled (+ optional 2× latent upscaler)
First run downloads ~27 GB to the Drive HF cache. Log goes to a file (01c lesson: never an unread
PIPE).

In [ ]:
import torch, os, logging
from diffusers import LTXConditionPipeline, LTXLatentUpsamplePipeline
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from diffusers.pipelines.ltx.modeling_latent_upsampler import LTXLatentUpsamplerModel

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {torch.cuda.get_device_name(0)}  ~{vram_gb:.0f} GB')

LOG = '/content/ltx_load.log'
logging.basicConfig(filename=LOG, level=logging.INFO)

MODEL_ID = 'Lightricks/LTX-Video-0.9.8-13B-distilled'
pipeline = LTXConditionPipeline.from_pretrained(MODEL_ID, dtype=torch.bfloat16)
pipeline.vae.enable_tiling()

if vram_gb >= 60:
    pipeline.to('cuda')
    print('Strategy: resident (>=60GB)')
elif vram_gb >= 30:
    from diffusers.hooks import apply_group_offloading
    onload, offload = torch.device('cuda'), torch.device('cpu')
    pipeline.transformer.enable_group_offload(onload_device=onload, offload_device=offload,
                                              offload_type='leaf_level', use_stream=True)
    apply_group_offloading(pipeline.text_encoder, onload_device=onload, offload_device=offload,
                           offload_type='block_level', num_blocks_per_group=2)
    apply_group_offloading(pipeline.vae, onload_device=onload, offload_device=offload,
                           offload_type='leaf_level')
    print('Strategy: group offloading (30-60GB)')
else:
    pipeline.enable_model_cpu_offload()
    print('Strategy: CPU offload (<30GB) — expect slower; consider 2B variant (§8)')

# Optional 2x latent upscaler (docs recipe: generate at 2/3 res, upscale, 5-step refine)
pipe_upsample = None
if UPSCALE_2X:
    try:
        upsampler = LTXLatentUpsamplerModel.from_pretrained(
            'a-r-r-o-w/LTX-0.9.8-Latent-Upsampler', dtype=torch.bfloat16)
        pipe_upsample = LTXLatentUpsamplePipeline(vae=pipeline.vae,
                                                  latent_upsampler=upsampler).to(torch.bfloat16)
        print('✅ Latent upscaler loaded.')
    except Exception as e:
        print('Upscaler failed to load, will run single-pass:', str(e)[:150])

print('✅ LTX pipeline ready.  Log →', LOG)

## 4. The `interpolate()` helper (multi-keyframe)
Distilled-model settings straight from the official docs: `guidance_scale=1.0`,
`guidance_rescale=0.7`, custom timesteps, VAE dims rounded to multiples of 32.

Keyframes are `LTXVideoCondition(image=..., frame_index=N)` where `frame_index` is the position in
the final clip (0 = first, `NUM_FRAMES-1` = last, middle = `int(fraction * (NUM_FRAMES-1))`).
**Tip (from the docs):** use *similar* images for best results — big subject/lighting divergence
between keyframes causes abrupt transitions.

In [ ]:
import torch, time
from pathlib import Path
from PIL import Image
from diffusers.utils import export_to_video

def _round32(x):
    return int(x) // 32 * 32

def interpolate(start_frame, prompt, end_frame=None, middle_frames=None,
                width=WIDTH, height=HEIGHT, num_frames=NUM_FRAMES,
                seed=0, upscale=UPSCALE_2X, tag=''):
    """
    Generate a clip between keyframes.
      start_frame  : PIL image or path (required)
      end_frame    : PIL image or path (optional; omit for plain I2V)
      middle_frames: list of (path|PIL, fraction) e.g. [('mid.png', 0.5)]
      prompt       : describe MOTION/action; identity comes from the frames themselves.
    Returns path to the saved .mp4.
    """
    def load(p):
        return p if isinstance(p, Image.Image) else Image.open(p).convert('RGB')

    conditions = [LTXVideoCondition(image=load(start_frame), frame_index=0)]
    for path, frac in (middle_frames or []):
        conditions.append(LTXVideoCondition(image=load(path),
                                            frame_index=int(frac * (num_frames - 1))))
    if end_frame is not None:
        conditions.append(LTXVideoCondition(image=load(end_frame), frame_index=num_frames - 1))

    neg = 'worst quality, inconsistent motion, blurry, jittery, distorted'
    generator = torch.Generator().manual_seed(seed)

    h, w = _round32(height), _round32(width)

    if upscale and pipe_upsample is not None:
        # docs recipe: 2/3 res → 2x latent upscale → 5-step refine → resize to target
        dh, dw = _round32(h * 2 / 3), _round32(w * 2 / 3)
        latents = pipeline(
            conditions=conditions, prompt=prompt, negative_prompt=neg,
            width=dw, height=dh, num_frames=num_frames,
            timesteps=[1000, 993, 987, 981, 975, 909, 725, 0.03],
            decode_timestep=0.05, decode_noise_scale=0.025, image_cond_noise_scale=0.0,
            guidance_scale=1.0, guidance_rescale=0.7,
            generator=generator, output_type='latent',
        ).frames
        uh, uw = dh * 2, dw * 2
        upscaled = pipe_upsample(latents=latents, adain_factor=1.0,
                                 tone_map_compression_ratio=0.6,
                                 output_type='latent').frames
        frames = pipeline(
            conditions=conditions, prompt=prompt, negative_prompt=neg,
            width=uw, height=uh, num_frames=num_frames,
            denoise_strength=0.999, timesteps=[1000, 909, 725, 421, 0],
            latents=upscaled, decode_timestep=0.05, decode_noise_scale=0.025,
            image_cond_noise_scale=0.0, guidance_scale=1.0, guidance_rescale=0.7,
            generator=generator, output_type='pil',
        ).frames[0]
        frames = [f.resize((w, h)) for f in frames]
    else:
        frames = pipeline(
            conditions=conditions, prompt=prompt, negative_prompt=neg,
            width=w, height=h, num_frames=num_frames,
            timesteps=[1000, 993, 987, 981, 975, 909, 725, 0.03],
            decode_timestep=0.05, decode_noise_scale=0.025, image_cond_noise_scale=0.0,
            guidance_scale=1.0, guidance_rescale=0.7,
            generator=generator, output_type='pil',
        ).frames[0]

    ts = time.strftime('%Y%m%d_%H%M%S')
    out = Path(VID_OUT) / f'{ts}_{tag}.mp4' if tag else Path(VID_OUT) / f'{ts}.mp4'
    export_to_video(frames, str(out), fps=FPS)
    print(f'✅ clip → {out}  ({num_frames} frames @ {FPS} fps, {w}x{h})')
    return str(out)

print('interpolate() ready.')

## 5. (Optional) upload keyframes, or point at 02a outputs

In [ ]:
# Option A: use stills you already generated in 02a — browse the folders:
import glob
for d in sorted(glob.glob(f'{STILLS_ROOT}/*'))[-3:]:
    print(d)
    for p in sorted(glob.glob(f'{d}/*.png')):
        print('   ', p)

# Option B: upload new keyframes from this notebook
# from google.colab import files
# up = files.upload()
# START = list(up)[0]  # etc.

# Set your actual paths here (Option A or B):
START = None    # e.g. f'{STILLS_ROOT}/20260906_120000/00_s1000.png'
END   = None    # e.g. f'{STILLS_ROOT}/20260906_120000/02_s1194.png'  (or None for I2V)
print('Set START (and optionally END) below in the run cell.')

## 6. Run a clip

In [ ]:
# Motion prompt: describe the ACTION/CAMERA, not the character (frames carry identity).
MOTION_PROMPT = ("She looks up slowly, hair shifting in a light breeze, subtle smile. "
                 "Slow cinematic push-in, shallow depth of field, soft natural light.")

assert START is not None, 'Set START in cell 5 first.'

clip = interpolate(
    start_frame=START,
    end_frame=END,                       # None → image-to-video (start frame only)
    middle_frames=None,                  # e.g. [('mid.png', 0.5)] for a mid keyframe
    prompt=f'{TRIGGER_TOKEN}, {MOTION_PROMPT}',
    seed=0,
    tag='flf2v',
)

from IPython.display import Video, display
display(Video(clip, width=720))

## 7. Chaining clips (longer sequences)
The spec 03 chaining rule: last frame of clip N → start frame of clip N+1. Extract the last frame
with ffmpeg, feed it back into `interpolate()`.

In [ ]:
import subprocess
from pathlib import Path

def last_frame_of(mp4, out_png):
    Path(out_png).parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['ffmpeg', '-y', '-sseof', '-0.1', '-i', mp4, '-frames:v', '1', out_png],
                   check=True, capture_output=True)
    return out_png

# Example: extend the clip from §6 by one more segment
# nxt_start = last_frame_of(clip, f'{VID_OUT}/chain/last_frame.png')
# clip2 = interpolate(start_frame=nxt_start, end_frame=END3,
#                     prompt=f'{TRIGGER_TOKEN}, she walks forward through the doorway', tag='seg2')
# Then stitch: ffmpeg concat (see 05_agentic_video.ipynb for the full stitcher)

print('Chaining helpers ready.')

## 8. Commented alternates (bigger/faster/cheaper — try later)
All the other Mode 1 LTX variants we may want, per spec 03's model table.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# A) LTX-Video 2B (base, guidance-distilled 50 steps) — T4 / free tier, ~6-8 GB
#    The main "Lightricks/LTX-Video" repo carries the 2B variant; use the
#    I2V pipeline for single-start-frame generation.
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import LTXImageToVideoPipeline
# from diffusers.utils import load_image, export_to_video
# pipe2b = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video",
#     dtype=torch.bfloat16); pipe2b.to('cuda')
# img = load_image(START)
# out = pipe2b(image=img, prompt=MOTION_PROMPT, width=832, height=480, num_frames=161,
#              decode_timestep=0.03, decode_noise_scale=0.025,
#              num_inference_steps=50, guidance_scale=5.0).frames[0]
# export_to_video(out, f'{VID_OUT}/ltx2b.mp4', fps=24)

# ─────────────────────────────────────────────────────────────────────────
# B) GGUF single-file 2B Q3 — runs in ~6 GB (16 GB card / L4 comfortable)
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import LTXPipeline, AutoModel, GGUFQuantizationConfig
# transformer = AutoModel.from_single_file(
#     "https://huggingface.co/city96/LTX-Video-gguf/blob/main/ltx-video-2b-v0.9-Q3_K_S.gguf",
#     quantization_config=GGUFQuantizationConfig(compute_dtype=torch.bfloat16),
#     dtype=torch.bfloat16)
# pipe = LTXPipeline.from_pretrained("Lightricks/LTX-Video", transformer=transformer,
#                                    dtype=torch.bfloat16)

# ─────────────────────────────────────────────────────────────────────────
# C) 0.9.7-dev + ltxv-spatial-upscaler-0.9.7 (pre-0.9.8 recipe, higher quality
#    ceiling, not distilled → 30 steps + guidance 5.0)
# ─────────────────────────────────────────────────────────────────────────
# pipeline7 = LTXConditionPipeline.from_pretrained("Lightricks/LTX-Video-0.9.7-dev",
#                                                  dtype=torch.bfloat16)
# pipe_up7 = LTXLatentUpsamplePipeline.from_pretrained("Lightricks/ltxv-spatial-upscaler-0.9.7",
#                                                      vae=pipeline7.vae, dtype=torch.bfloat16)
# ... same 4-stage recipe as §4 but timesteps=None, num_inference_steps=30,
#     guidance_scale=5.0, guidance_rescale=0.7

# ─────────────────────────────────────────────────────────────────────────
# D) fp8 layerwise casting for the 13B transformer (saves ~13 GB, tiny speed hit)
#    Use this on a 40GB A100 if group offloading feels slow.
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import AutoModel
# transformer = AutoModel.from_pretrained("Lightricks/LTX-Video-0.9.8-13B-distilled",
#     subfolder="transformer", dtype=torch.bfloat16)
# transformer.enable_layerwise_casting(storage_dtype=torch.float8_e4m3fn,
#                                      compute_dtype=torch.bfloat16)
# pipeline = LTXConditionPipeline.from_pretrained(MODEL_ID, transformer=transformer,
#                                                 dtype=torch.bfloat16)

# ─────────────────────────────────────────────────────────────────────────
# E) LTX CHARACTER LoRA — once trained (musubi-tuner or an LTX LoRA trainer), load with:
# ─────────────────────────────────────────────────────────────────────────
# pipeline.load_lora_weights("path/or/repo/of/ltx-character-lora", adapter_name="yuna")
# pipeline.set_adapters("yuna")
# and prefix prompts with the LoRA's trigger word.

print('Section 8: alternates commented out — pick one when needed.')

## 9. Log to metadata

In [ ]:
import json, os, glob, time
meta_path = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}/metadata.json'
os.makedirs(os.path.dirname(meta_path), exist_ok=True)
meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {'name': CHARACTER_NAME}

clips = sorted(glob.glob(f'{VID_OUT}/*.mp4'))
meta.setdefault('video_log', []).append({
    'ts': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'mode': 'mode1_ltx',
    'model': 'LTX-Video-0.9.8-13B-distilled',
    'clips': clips[-5:],
})
json.dump(meta, open(meta_path, 'w'), indent=2)
print(f'metadata.json updated — {len(clips)} clips total in {VID_OUT}')
print('\n✅ 03a complete. High-quality FLF2V: 03b (Wan FLF2V). Agentic chaining: 05.')